# Fase 17E-0 — Gate de agregação híbrida cifrada

**Campanha:** `THESIS_OFFICIAL_CAMPAIGN_V2_20260901`  
**Classificação:** validação técnica pré-campanha — **não usar como resultado final da dissertação**.

Este notebook prova que cinco atualizações são transcodificadas por Rubato→CKKS, ponderadas e somadas em CKKS, e que existe somente uma decifragem, depois da agregação. Valida os comprimentos oficiais Dahl `13`, PhysioNet Challenge 2012 `266` e CheXchoNet `513`.

> Execute as células em ordem. Não altere commit, seed, número de clientes, variante Rubato ou tolerâncias.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, re, shutil, subprocess, time, hashlib, numpy as np

CAMPAIGN_ID='THESIS_OFFICIAL_CAMPAIGN_V2_20260901'
BASE=Path('/content/drive/MyDrive/Mestrado_Criptografia/OFFICIAL_CAMPAIGN_V2')
CAMPAIGN_ROOT=BASE/CAMPAIGN_ID
CONTROL=CAMPAIGN_ROOT/'00_CAMPAIGN_CONTROL'
OUT=CAMPAIGN_ROOT/'02_SMOKE_TESTS'/'PHASE17E0_HYBRID_AGGREGATION_GATE'
OUT.mkdir(parents=True,exist_ok=True); CONTROL.mkdir(parents=True,exist_ok=True)
print('CAMPAIGN_ID=',CAMPAIGN_ID); print('RESULT_FOLDER=',OUT)


## 1. Preparação reprodutível do RtF

In [ ]:
REPO=Path('/content/RtF-Transciphering')
COMMIT='105fc73115b56f1d6ff357029c7682b19a6d8510'
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','golang-go','git'],check=True)
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/KAIST-CryptLab/RtF-Transciphering.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','--all','--tags'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert actual==COMMIT,(actual,COMMIT)
print(subprocess.check_output(['go','version'],text=True).strip()); print('COMMIT_OK=',actual)


## 2. Instalação e auditoria estática do bridge agregado

In [ ]:
GO_SOURCE = 'package ckks_fv\n\nimport (\n    "crypto/rand"\n    "encoding/json"\n    "fmt"\n    "math"\n    "os"\n    "runtime"\n    "runtime/debug"\n    "testing"\n    "time"\n    "github.com/ldsec/lattigo/v2/utils"\n)\n\ntype phase17E0Request struct {\n    Dataset string `json:"dataset"`\n    OriginalLength int `json:"original_length"`\n    ClientDeltas [][]float64 `json:"client_deltas"`\n    ClientWeights []float64 `json:"client_weights"`\n}\ntype phase17E0Response struct {\n    Dataset string `json:"dataset"`\n    OriginalLength int `json:"original_length"`\n    Clients int `json:"clients"`\n    RecoveredAggregate []float64 `json:"recovered_aggregate"`\n    MaxAbsError float64 `json:"max_abs_error"`\n    MeanAbsError float64 `json:"mean_abs_error"`\n    MaxPaddingError float64 `json:"max_padding_error"`\n    WallSeconds float64 `json:"wall_seconds"`\n    RubatoVariant string `json:"rubato_variant"`\n    BridgeVersion string `json:"bridge_version"`\n    AggregatedBeforeDecrypt bool `json:"aggregated_before_decrypt"`\n    ClientDecryptions int `json:"client_decryptions"`\n}\n\nfunc TestPhase17E0EncryptedFedAvg(t *testing.T) {\n    raw := os.Getenv("PHASE17E0_REQUEST_JSON")\n    if raw == "" { t.Fatal("PHASE17E0_REQUEST_JSON ausente") }\n    var req phase17E0Request\n    if err := json.Unmarshal([]byte(raw), &req); err != nil { t.Fatal(err) }\n    if len(req.ClientDeltas) != 5 || len(req.ClientWeights) != 5 { t.Fatal("expected exactly 5 clients and 5 weights") }\n    if req.OriginalLength < 1 || req.OriginalLength > 513 { t.Fatal("original_length must be in [1,513]") }\n    sumW := 0.0\n    for i := range req.ClientDeltas {\n        if len(req.ClientDeltas[i]) != req.OriginalLength { t.Fatalf("client %d: expected %d values, got %d", i, req.OriginalLength, len(req.ClientDeltas[i])) }\n        if req.ClientWeights[i] <= 0 { t.Fatalf("client %d: weight must be positive", i) }\n        sumW += req.ClientWeights[i]\n    }\n    if math.Abs(sumW-1.0) > 1e-12 { t.Fatalf("weights must sum to 1; got %.17g", sumW) }\n\n    rubatoParam := RUBATO80S\n    blocksize := RubatoParams[rubatoParam].Blocksize\n    numRound := RubatoParams[rubatoParam].NumRound\n    plainModulus := RubatoParams[rubatoParam].PlainModulus\n    sigma := RubatoParams[rubatoParam].Sigma\n    hbtpParams := RtFRubatoParams[0]\n    params, err := hbtpParams.Params(); if err != nil { t.Fatal(err) }\n    params.SetPlainModulus(plainModulus); params.SetLogFVSlots(params.LogN())\n    messageScaling := float64(params.PlainModulus()) / hbtpParams.MessageRatio\n    rubatoModDown := RubatoModDownParams[rubatoParam].CipherModDown\n    stcModDown := RubatoModDownParams[rubatoParam].StCModDown\n    kgen := NewKeyGenerator(params); sk, pk := kgen.GenKeyPairSparse(hbtpParams.H)\n    fvEncoder := NewMFVEncoder(params); ckksEncoder := NewCKKSEncoder(params)\n    fvEncryptor := NewMFVEncryptorFromPk(params, pk); ckksDecryptor := NewCKKSDecryptor(params, sk)\n    rotationsHalfBoot := kgen.GenRotationIndexesForHalfBoot(params.LogSlots(), hbtpParams)\n    pDcds := fvEncoder.GenSlotToCoeffMatFV(2)\n    rotations := append(rotationsHalfBoot, kgen.GenRotationIndexesForSlotsToCoeffsMat(pDcds)...)\n    rotkeys := kgen.GenRotationKeysForRotations(rotations, true, sk); rlk := kgen.GenRelinearizationKey(sk)\n    evk := EvaluationKey{Rlk: rlk, Rtks: rotkeys}\n    hbtp, err := NewHalfBootstrapper(params, hbtpParams, BootstrappingKey{Rlk: rlk, Rtks: rotkeys}); if err != nil { t.Fatal(err) }\n    fvEvaluator := NewMFVEvaluator(params, evk, pDcds)\n    ckksEvaluator := NewCKKSEvaluator(params, evk)\n    key := make([]uint64, blocksize); for i := range key { key[i] = uint64(i+1) }\n    keyRubato := NewMFVRubato(rubatoParam, params, fvEncoder, fvEncryptor, fvEvaluator, rubatoModDown[0])\n    kCt := keyRubato.EncKey(key)\n\n    start := time.Now()\n    expected := make([]float64, 513)\n    var encryptedAggregate *Ciphertext\n    for clientID := 0; clientID < 5; clientID++ {\n        fmt.Printf("PHASE17E0_PROGRESS dataset=%s client=%d/5 stage=transciphering\\n", req.Dataset, clientID+1)\n        data := make([]float64, params.N()); copy(data[:req.OriginalLength], req.ClientDeltas[clientID])\n        for j := 0; j < req.OriginalLength; j++ { expected[j] += req.ClientWeights[clientID] * req.ClientDeltas[clientID][j] }\n        nonces := make([][]byte, params.N()); keystream := make([][]uint64, params.N())\n        for i := 0; i < params.N(); i++ { nonces[i] = make([]byte,8); if _,err=rand.Read(nonces[i]);err!=nil{t.Fatal(err)} }\n        counter := make([]byte,8); if _,err=rand.Read(counter);err!=nil{t.Fatal(err)}\n        for i := 0; i < params.N(); i++ { keystream[i] = plainRubato(blocksize,numRound,nonces[i],counter,key,plainModulus,sigma) }\n        coeffs := make([]float64, params.N())\n        for i := 0; i < params.N()/2; i++ { j:=utils.BitReverse64(uint64(i),uint64(params.LogN()-1)); coeffs[j]=data[i]; coeffs[j+uint64(params.N()/2)]=data[i+params.N()/2] }\n        plainRingT := ckksEncoder.EncodeCoeffsRingTNew(coeffs,messageScaling); poly:=plainRingT.Value()[0]\n        for i := 0; i < params.N(); i++ { j:=utils.BitReverse64(uint64(i),uint64(params.LogN())); poly.Coeffs[0][j]=(poly.Coeffs[0][j]+keystream[i][0])%params.PlainModulus() }\n        plaintext:=NewPlaintextFVLvl(params,0); fvEncoder.FVScaleUp(plainRingT,plaintext)\n        // A fresh Rubato workspace per client prevents mutable modulus-chain state from leaking across clients.\n        clientRubato:=NewMFVRubato(rubatoParam,params,fvEncoder,fvEncryptor,fvEvaluator,rubatoModDown[0])\n        fvKeystreams:=clientRubato.Crypt(nonces,counter,kCt,rubatoModDown); fvKS:=fvEvaluator.SlotsToCoeffs(fvKeystreams[0],stcModDown)\n        level:=fvKS.Level()\n        if level>0 { fvEvaluator.ModSwitchMany(fvKS,fvKS,level) }\n        ciphertext:=NewCiphertextFVLvl(params,1,0); ciphertext.Value()[0]=plaintext.Value()[0].CopyNew(); fvEvaluator.Sub(ciphertext,fvKS,ciphertext); fvEvaluator.TransformToNTT(ciphertext,ciphertext)\n        ciphertext.SetScale(math.Exp2(math.Round(math.Log2(float64(params.Qi()[0])/float64(params.PlainModulus())*messageScaling))))\n        ctBoot,_:=hbtp.HalfBoot(ciphertext,false)\n        weighted:=ckksEvaluator.MultByConstNew(ctBoot,req.ClientWeights[clientID])\n        if encryptedAggregate==nil { encryptedAggregate=weighted } else { ckksEvaluator.Add(encryptedAggregate,weighted,encryptedAggregate) }\n        fmt.Printf("PHASE17E0_PROGRESS dataset=%s client=%d/5 stage=encrypted_aggregate_updated\\n", req.Dataset, clientID+1)\n        // Release all client-local material before the next transciphering. Only the CKKS aggregate survives.\n        data=nil; nonces=nil; keystream=nil; coeffs=nil; plainRingT=nil; plaintext=nil\n        clientRubato=nil; fvKeystreams=nil; fvKS=nil; ciphertext=nil; ctBoot=nil; weighted=nil\n        runtime.GC(); debug.FreeOSMemory()\n        fmt.Printf("PHASE17E0_PROGRESS dataset=%s client=%d/5 stage=memory_released\\n", req.Dataset, clientID+1)\n    }\n    // SECURITY BOUNDARY: the sole decrypt operation is after all five CKKS ciphertexts were aggregated.\n    values:=ckksEncoder.DecodeComplex(ckksDecryptor.DecryptNew(encryptedAggregate),params.LogSlots())\n    recovered:=make([]float64,req.OriginalLength); maxErr:=0.0; meanErr:=0.0; maxPad:=0.0\n    for i:=0;i<req.OriginalLength;i++ { recovered[i]=real(values[i]); e:=math.Abs(recovered[i]-expected[i]); meanErr+=e; if e>maxErr{maxErr=e} }\n    meanErr/=float64(req.OriginalLength)\n    for i:=req.OriginalLength;i<513;i++ { e:=math.Abs(real(values[i])); if e>maxPad{maxPad=e} }\n    resp:=phase17E0Response{req.Dataset,req.OriginalLength,5,recovered,maxErr,meanErr,maxPad,time.Since(start).Seconds(),"RUBATO80S","phase17e0_encrypted_fedavg_v1",true,0}\n    b,err:=json.Marshal(resp);if err!=nil{t.Fatal(err)};fmt.Printf("PHASE17E0_JSON:%s\\n",b)\n}\n'
GO_FILE=REPO/'ckks_fv'/'phase17e0_encrypted_fedavg_test.go'
GO_FILE.write_text(GO_SOURCE,encoding='utf-8')
decrypt_positions=[m.start() for m in re.finditer(r'DecryptNew\(',GO_SOURCE)]
boundary=GO_SOURCE.index('SECURITY BOUNDARY')
assert len(decrypt_positions)==1,'Deve existir exatamente uma decifragem.'
assert decrypt_positions[0]>boundary,'A decifragem apareceu antes da fronteira de agregação.'
assert 'RecoveredDelta' not in GO_SOURCE and 'recovered_delta' not in GO_SOURCE
assert 'NewCKKSEvaluator' in GO_SOURCE and 'MultByConstNew' in GO_SOURCE and 'ckksEvaluator.Add' in GO_SOURCE
source_sha=hashlib.sha256(GO_SOURCE.encode()).hexdigest()
subprocess.run(['gofmt','-w',str(GO_FILE)],check=True)
print('STATIC_SECURITY_GATE=PASS'); print('SOURCE_SHA256=',source_sha)


## 3. Compilação isolada

In [ ]:
t0=time.time()
p=subprocess.run(['go','test','./ckks_fv','-run','^$','-count=1'],cwd=REPO,text=True,capture_output=True)
print(p.stdout); print(p.stderr)
assert p.returncode==0,'Falha de compilação do bridge.'
print('COMPILE_GATE=PASS seconds=',round(time.time()-t0,2))


## 4. Validação criptográfica dos três adaptadores

Esta etapa é longa. Cada linha `PHASE17E0_PROGRESS` confirma o cliente e o estágio atual. O notebook nunca imprime os vetores completos.


In [ ]:
from collections import deque
TARGETS=[('DAHL_RATS',13),('PHYSIONET_CHALLENGE_2012',266),('CHEXCHONET',513)]
TOL_MAX=1e-3; TOL_MEAN=1e-4; TOL_PADDING=1e-3
results=[]
for dataset,n in TARGETS:
    checkpoint=OUT/f'{dataset}_PHASE17E0_RESULT.json'
    if checkpoint.exists():
        saved=json.loads(checkpoint.read_text(encoding='utf-8'))
        if saved.get('approved') is True and saved.get('bridge_source_sha256')==source_sha:
            print('CHECKPOINT APROVADO REUTILIZADO:',dataset); results.append(saved); continue
    print('='*100); print('INICIANDO',dataset,'modelo',n,'→ bridge 513; 5 clientes')
    rng=np.random.default_rng(42+n)
    deltas=rng.normal(0,0.02,size=(5,n)).astype(float)
    sample_counts=np.array([101,173,229,307,419],dtype=float); weights=sample_counts/sample_counts.sum()
    req={'dataset':dataset,'original_length':n,'client_deltas':deltas.tolist(),'client_weights':weights.tolist()}
    env=os.environ.copy(); env['PHASE17E0_REQUEST_JSON']=json.dumps(req,separators=(',',':')); env['GOGC']='20'
    proc=subprocess.Popen(['go','test','./ckks_fv','-run','^TestPhase17E0EncryptedFedAvg$','-count=1','-v','-timeout=0'],cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
    payload=None; tail=deque(maxlen=120); log_path=OUT/f'{dataset}_PHASE17E0_GO.log'
    log_file=log_path.open('w',encoding='utf-8')
    for line in proc.stdout:
        line=line.rstrip()
        log_file.write(line+'\n'); log_file.flush(); tail.append(line)
        if 'PHASE17E0_PROGRESS' in line or '--- FAIL' in line: print(line,flush=True)
        if 'PHASE17E0_JSON:' in line: payload=json.loads(line.split('PHASE17E0_JSON:',1)[1])
    rc=proc.wait(); log_file.close()
    if rc!=0 or payload is None:
        print('\nÚLTIMAS LINHAS DO PROCESSO GO:'); print('\n'.join(tail))
        raise RuntimeError(f'Bridge falhou para {dataset}, rc={rc}. Log completo: {log_path}')
    expected=np.average(deltas,axis=0,weights=weights)
    recovered=np.array(payload['recovered_aggregate'])
    independent_max=float(np.max(np.abs(recovered-expected)))
    checks={'five_clients':payload['clients']==5,'aggregated_before_decrypt':payload['aggregated_before_decrypt'] is True,'zero_client_decryptions':payload['client_decryptions']==0,'length_ok':len(recovered)==n,'max_error_ok':independent_max<=TOL_MAX,'mean_error_ok':payload['mean_abs_error']<=TOL_MEAN,'padding_ok':payload['max_padding_error']<=TOL_PADDING}
    payload['independent_max_abs_error']=independent_max; payload['checks']=checks; payload['approved']=all(checks.values()); payload['bridge_source_sha256']=source_sha
    checkpoint.write_text(json.dumps(payload,indent=2),encoding='utf-8'); results.append(payload)
    print(json.dumps({'dataset':dataset,'approved':payload['approved'],'wall_seconds':payload['wall_seconds'],'max_abs_error':independent_max,'mean_abs_error':payload['mean_abs_error'],'max_padding_error':payload['max_padding_error'],'checks':checks},indent=2))
    assert payload['approved'],f'Gate reprovado: {dataset}'


## 5. Gate mestre, persistência e pacote de evidências

In [ ]:
gate={'phase':'17E0','campaign_id':CAMPAIGN_ID,'classification':'PRE_CAMPAIGN_TECHNICAL_VALIDATION','usable_as_final_thesis_result':False,'bridge_commit':COMMIT,'bridge_source_sha256':source_sha,'rubato_variant':'RUBATO80S','clients':5,'aggregation_domain':'CKKS_CIPHERTEXT','decryptions_before_aggregation':0,'datasets':{r['dataset']:r for r in results},'approved':all(r['approved'] for r in results),'next_authorized_step':'FASE_17E_HYBRID_SMOKE_TESTS' if all(r['approved'] for r in results) else None}
assert gate['approved']
gate_path=CONTROL/'PHASE17E0_MASTER_GATE.json'; gate_path.write_text(json.dumps(gate,indent=2),encoding='utf-8')
(OUT/'PHASE17E0_MASTER_GATE.json').write_text(json.dumps(gate,indent=2),encoding='utf-8')
(OUT/'phase17e0_encrypted_fedavg_test.go').write_text(GO_FILE.read_text(),encoding='utf-8')
readme='Fase 17E-0: validação técnica pré-campanha. Não usar métricas como resultados finais da dissertação.\nGate: '+str(gate['approved'])+'\n'
(OUT/'README.txt').write_text(readme,encoding='utf-8')
archive=shutil.make_archive('/content/PHASE17E0_EVIDENCE','zip',OUT.parent,OUT.name)
drive_zip=OUT/'PHASE17E0_EVIDENCE.zip'; shutil.copy2(archive,drive_zip)
print('='*100); print(json.dumps({'PHASE17E0_APPROVED':gate['approved'],'training_authorized':False,'next_authorized_step':gate['next_authorized_step'],'gate':str(gate_path),'evidence_zip':str(drive_zip)},indent=2)); print('='*100)


## Critério de encerramento

Somente a mensagem `PHASE17E0_APPROVED=true` autoriza criar/executar a Fase 17E. A Fase 17E-0 não contém treino nem produz métricas finais da dissertação.
